# Pipeline Engineering (scikit-learn)

When beginners write machine learning code, it usually looks like a messy recipe: they impute missing numbers, then scale them, then one-hot encode the text, then combine everything back together, and finally train a model. 

This creates "Spaghetti Code." It is incredibly difficult to read, nearly impossible to reuse on new data, and highly prone to Data Leakage.

Scikit-Learn provides two brilliant tools to fix this:
1. **`ColumnTransformer`**: To route different columns (like text vs. numbers) through different preprocessing steps.
2. **`Pipeline`**: To chain the entire preprocessing and modeling workflow into one single, unbroken object.

Let's set up a Python sandbox with a messy dataset featuring missing numbers, missing text, and mixed data types!

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Create a messy, mixed-type dataset
data = {
    'age': [25, 32, np.nan, 45, 22, 38, 50, np.nan, 28, 41],
    'income': [50000, 65000, 120000, np.nan, 45000, 80000, 150000, 55000, 60000, 95000],
    'city': ['New York', 'London', 'Paris', 'London', np.nan, 'New York', 'Paris', 'London', 'New York', np.nan],
    # The Target: Did they buy the product?
    'purchased': [0, 1, 1, 0, 0, 1, 1, 0, 0, 1] 
}

df = pd.DataFrame(data)

# 1. ALWAYS split the data first!
X = df.drop(columns=['purchased'])
y = df['purchased']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("✅ Data successfully split into Train and Test sets.")
display(X_train)

✅ Data successfully split into Train and Test sets.


,age,income,city
5,38.0,80000.0,New York
0,25.0,50000.0,New York
7,NaN,55000.0,London
2,NaN,120000.0,Paris
9,41.0,95000.0,NaN
4,22.0,45000.0,NaN
3,45.0,NaN,London
6,50.0,150000.0,Paris


# 1. Building Sub-Pipelines (The Preprocessors)
Numerical data and Categorical data require completely different treatments. 
* **Numbers** need to have missing values filled with the Median, and then be Scaled.
* **Categories** need to have missing values filled with a placeholder (like "Missing"), and then be One-Hot Encoded.

We can create mini-pipelines for each data type.

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. The Numerical Mini-Pipeline
# It does Step 1: Impute, then Step 2: Scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 2. The Categorical Mini-Pipeline
# It does Step 1: Impute, then Step 2: One-Hot Encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) 
    # handle_unknown='ignore' protects us if a brand new city appears in the future!
])

print("✅ Mini-pipelines created for numbers and categories.")

✅ Mini-pipelines created for numbers and categories.


# 2. The `ColumnTransformer` (The Traffic Cop)
Now that we have our mini-pipelines, we need to tell Scikit-Learn which columns to apply them to. The `ColumnTransformer` acts as a traffic cop, routing the data to the correct pipeline.

In [3]:
from sklearn.compose import ColumnTransformer

# Identify our column names
numeric_features = ['age', 'income']
categorical_features = ['city']

# Create the ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("✅ ColumnTransformer ready to route data.")

✅ ColumnTransformer ready to route data.


# 3. The Master Pipeline (End-to-End)
Finally, we combine our preprocessing traffic cop with our actual Machine Learning model into one Master Pipeline.

In [4]:
from sklearn.ensemble import RandomForestClassifier

# Combine the preprocessor and the model
master_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Now, we train the ENTIRE process with ONE single line of code!
master_pipeline.fit(X_train, y_train)

print("✅ Master Pipeline successfully trained!")

✅ Master Pipeline successfully trained!


# 4. Painless Predictions and Safety
Why did we do all this work? Because predicting new, unseen data is now incredibly safe and takes exactly one line of code.

When you call `master_pipeline.predict(X_test)`, the pipeline automatically takes the raw `X_test` data, routes the numbers to be imputed and scaled, routes the text to be imputed and encoded, recombines them, and hands them to the Random Forest model. 

In [5]:
# Predict on the raw, uncleaned Testing data
predictions = master_pipeline.predict(X_test)

print("--- Test Set Predictions ---")
print(f"Raw Input Data:\n{X_test}\n")
print(f"Model Predictions: {predictions}")
print(f"Actual Answers:    {y_test.values}")

--- Test Set Predictions ---
Raw Input Data:
    age   income      city
8  28.0  60000.0  New York
1  32.0  65000.0    London

Model Predictions: [0 0]
Actual Answers:    [0 1]


# 5. Cross-Validation (The Holy Grail)
If you want to test how good your model is using Cross-Validation (`cross_val_score`), you **must** use a Pipeline. 

Cross-validation chops your training data into 5 different pieces, trains on 4, and tests on 1. If you preprocessed your data *before* chopping it, you caused Data Leakage across all 5 folds! By passing the Pipeline into the cross-validator, Scikit-Learn ensures that the data is split first, and *then* preprocessed perfectly for every single fold.

In [6]:
from sklearn.model_selection import cross_val_score

# Evaluate our pipeline using 3-fold cross-validation
# (We use 3 because our toy dataset is very small)
scores = cross_val_score(master_pipeline, X, y, cv=3)

print("\n--- Cross-Validation Results ---")
print(f"Accuracy Scores for each fold: {scores}")
print(f"Average Accuracy: {scores.mean() * 100:.2f}%")


--- Cross-Validation Results ---
Accuracy Scores for each fold: [0.75       0.33333333 1.        ]
Average Accuracy: 69.44%


## Real-World Use Case or Analogy:
Think of a Scikit-Learn Pipeline like an **Automated Car Manufacturing Plant**:

* **Spaghetti Code (The Old Way)**: A worker carries a piece of raw steel across the factory to the painting station, then carries it to the engine station, realizes they forgot to polish it, carries it back, and eventually bolts it to the frame. It's chaotic, error-prone, and relies entirely on human memory. 
* **The Mini-Pipelines (`numeric_transformer`)**: You build specialized conveyor belts. One belt is only for metal components (welding, polishing). Another belt is only for electronics (wiring, testing).
* **The Traffic Cop (`ColumnTransformer`)**: The sorting machine at the front door. When a delivery truck dumps raw materials, the sorter perfectly places the metal on the metal belt, and the wires on the electronics belt.
* **The Master Pipeline**: The entire factory floor. You put raw, messy steel and plastic into the front door (`pipeline.predict(raw_data)`). You press one button, the conveyor belts run, and a fully functional, flawless car drives out the back door. Once built, you can pick up this entire factory and place it anywhere in the world to guarantee the exact same results every time.

---